# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Szan-12345/FLYRANK-MACHINE-LEARNING/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
%pip -q install duckdb huggingface_hub


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)

clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [8]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [13]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [14]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.543     0.336     0.415      9389
           1      0.684     0.835     0.752     16162

    accuracy                          0.652     25551
   macro avg      0.614     0.586     0.584     25551
weighted avg      0.632     0.652     0.628     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


**a. What one row means for my lane**
One row = one content item's Search Console performance on one day
(client_hash_id + content_hash_id + report_date) in fact_content_daily_performance.
My unit of analysis is one content item, as of a decision day.

**b. Which table(s) I use**
fact_content_daily_performance, sliced to month=2026-03. I leave fact_query_90d
out of this notebook — its 90-day trailing window isn't confirmed to align with
"as of March 15," so including it here risks a window-alignment leak (see limitation).

**c. Time window**
March 2026, split at the midpoint: days 1–15 = feature window (decision moment),
days 16–31 = outcome window the label is drawn from.

**d. What I'd predict or rank**
is_declining = 1 if summed impressions in days 16–31 < 80% of summed impressions
in days 1–15, else 0 — same proxy as notebook 02's trend_direction == 'down'.

**e. One thing I deliberately exclude**
fact_daily_sample (June 2026, the sealed final month) — never touched for
features or labels, only for a true holdout later.

## 3) Three queries, five features, the trap
**Query 1 — grain**

In [15]:
sample_pair = con.sql(f"""
    SELECT client_hash_id, content_hash_id
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    LIMIT 1
""").df().iloc[0]

grain_check = con.sql(f"""
    SELECT report_date, COUNT(*) AS rows_on_this_date
    FROM {TABLES['fact_daily']}
    WHERE client_hash_id = '{sample_pair.client_hash_id}'
      AND content_hash_id = '{sample_pair.content_hash_id}'
      AND report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY report_date
    ORDER BY report_date
""").df()

print(f"rows in March: {len(grain_check)}")
print(f"max rows on any single date: {grain_check['rows_on_this_date'].max()}  (should be 1)")
grain_check.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows in March: 31
max rows on any single date: 1  (should be 1)


,report_date,rows_on_this_date
0,2026-03-01,1
1,2026-03-02,1
2,2026-03-03,1
3,2026-03-04,1
4,2026-03-05,1
5,2026-03-06,1
6,2026-03-07,1
7,2026-03-08,1
8,2026-03-09,1
9,2026-03-10,1


**Query 3 — availability with IS TRUE**

In [16]:
desc_daily = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 0").df()
print(desc_daily[['column_name', 'column_type']].to_string(index=False))
bool_cols = desc_daily.loc[desc_daily['column_type'].str.upper() == 'BOOLEAN', 'column_name'].tolist()
print(f"\nBoolean columns found: {bool_cols}")

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

In [17]:
BOOLEAN_COLUMN = 'gsc_data_available'

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE {BOOLEAN_COLUMN} IS TRUE) AS available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

availability['pct_available'] = (availability['available_rows'] / availability['total_rows'] * 100).round(1)
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,pct_available
0,9841378,3611061,36.7


**Five features (days 1–15 only)**

In [18]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS impressions_h1,
        SUM(gsc_clicks)                                                 AS clicks_h1,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)         AS ctr_h1,
        AVG(gsc_avg_position)                                           AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days_h1
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()

print(f'{len(feature_frame):,} content items with a usable feature row')
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 content items with a usable feature row


,client_hash_id,content_hash_id,impressions_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,429.0,2.0,0.004662,4.247255,15
1,client_73cda7b4e4f265ea,content_05597932fe4da067,18.0,0.0,0.000000,4.939394,11
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,89.0,0.0,0.000000,3.010741,15
3,client_73cda7b4e4f265ea,content_05434271b257bb68,628.0,1.0,0.001592,5.330069,15
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,1280.0,9.0,0.007031,4.468441,15


1. impressions_h1 — sum of daily impressions, days 1–15. Knowable same-day, nothing from days 16–31.
2. clicks_h1 — sum of daily clicks, days 1–15. Same reasoning.
3. ctr_h1 — clicks_h1 / impressions_h1. Ratio of two already-knowable sums.
4. avg_position_h1 — mean daily rank position, days 1–15. Logged per report day.
5. active_days_h1 — count of days with impressions > 0 in the window. Just a count of days already past.

**The trap**

In [19]:
outcome = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_h2
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(outcome, on=['client_hash_id', 'content_hash_id'], how='inner')
data['is_declining'] = (data['impressions_h2'] < 0.8 * data['impressions_h1']).astype(int)
print(f'{len(data):,} rows | declining rate: {round(data["is_declining"].mean(), 3)}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 rows | declining rate: 0.296


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ['impressions_h1', 'clicks_h1', 'ctr_h1', 'avg_position_h1', 'active_days_h1']
model_data = data.dropna(subset=honest_features + ['is_declining'])
X, y = model_data[honest_features], model_data['is_declining']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f'HONEST score — AUC: {honest_auc:.3f}')

HONEST score — AUC: 0.588


In [21]:
# THE TRAP: add the label-derived column
leaky_features = honest_features + ['impressions_h2']
leaky_data = data.dropna(subset=leaky_features + ['is_declining'])
X_leak, y_leak = leaky_data[leaky_features], leaky_data['is_declining']
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leak, y_leak, test_size=0.25, random_state=42, stratify=y_leak)

leaky_model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_tr2, y_tr2)
leaky_auc = roc_auc_score(y_te2, leaky_model.predict_proba(X_te2)[:, 1])
print(f'LEAKY score — AUC: {leaky_auc:.3f}  <- jumps toward 1.0')
print(f'Jump: +{leaky_auc - honest_auc:.3f}')

LEAKY score — AUC: 1.000  <- jumps toward 1.0
Jump: +0.412


In [22]:
# REMOVE THE LEAK, KEEP THE HONEST NUMBER
del leaky_features
print(f'Final reported score (honest features only): AUC {honest_auc:.3f}')
print('impressions_h2 excluded — it is the quantity the label threshold is built from.')

Final reported score (honest features only): AUC 0.588
impressions_h2 excluded — it is the quantity the label threshold is built from.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [23]:
%pip -q install duckdb huggingface_hub

import os, getpass, duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:18} {n:>12,} rows")   # metadata-only — confirms the unit of analysis exists at the right scale

dim_clients                 104 rows
dim_content             519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily           78,835,655 rows
fact_daily_sample    11,694,072 rows
fact_query_90d        2,414,248 rows


**Unit of analysis:** one row = one content item's Search Console performance on one day
(`client_hash_id` + `content_hash_id` + `report_date`) in `fact_content_daily_performance`.

**Time window:** March 2026 — a mid-panel month, not the sealed `_sample` month (June 2026).
Split at the midpoint: days 1–15 = feature window (decision moment), days 16–31 = outcome window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [24]:
desc = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 0").df()
print(desc[["column_name", "column_type"]].to_string(index=False))

             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
              ai_copilot      BIGINT
 

**Table(s):** `fact_content_daily_performance`, sliced to `month=2026-03`. `fact_content_query_90d`
is left out — its 90-day trailing window isn't confirmed aligned to "as of March 15," so joining
it here risks a window-alignment leak.

**Label / proxy:** `is_declining` = 1 if summed impressions in days 16–31 < 80% of summed
impressions in days 1–15 — same shape as `w02`'s `trend_direction == "down"`, built fresh since
the warehouse has no pre-computed trend column.

| Field | Bucket | Why |
|---|---|---|
| `client_hash_id`, `content_hash_id` | Context | pseudonym IDs — join/group only |
| `report_date` | Context | defines the window split, not a model input |
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (days 1–15) | Feature | known as of the decision day |
| `gsc_data_available` | Context | filter flag, not a feature |
| `gsc_impressions` (days 16–31 / `impressions_h2`) | Label / proxy | the label is built from it — never a feature |
| `fact_content_query_90d` (all columns) | Excluded | window-alignment not confirmed — see above |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [25]:
sample_pair = con.sql(f"""
    SELECT client_hash_id, content_hash_id FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31' LIMIT 1
""").df().iloc[0]

grain_check = con.sql(f"""
    SELECT report_date, COUNT(*) AS rows_on_this_date
    FROM {TABLES['fact_daily']}
    WHERE client_hash_id = '{sample_pair.client_hash_id}'
      AND content_hash_id = '{sample_pair.content_hash_id}'
      AND report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY report_date ORDER BY report_date
""").df()
print(f"rows in March: {len(grain_check)}  |  max rows on any single date: {grain_check['rows_on_this_date'].max()} (should be 1)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows in March: 31  |  max rows on any single date: 1 (should be 1)


Three queries, on `month=2026-03`, then the five-feature frame, then the trap.

In [26]:
sample_pair = con.sql(f"""
    SELECT client_hash_id, content_hash_id FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31' LIMIT 1
""").df().iloc[0]

grain_check = con.sql(f"""
    SELECT report_date, COUNT(*) AS rows_on_this_date
    FROM {TABLES['fact_daily']}
    WHERE client_hash_id = '{sample_pair.client_hash_id}'
      AND content_hash_id = '{sample_pair.content_hash_id}'
      AND report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY report_date ORDER BY report_date
""").df()
print(f"rows in March: {len(grain_check)}  |  max rows on any single date: {grain_check['rows_on_this_date'].max()} (should be 1)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows in March: 31  |  max rows on any single date: 1 (should be 1)


In [27]:
con.sql(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_content_items,n_clients,first_date,last_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [28]:
bool_cols = desc.loc[desc["column_type"].str.upper() == "BOOLEAN", "column_name"].tolist()
print("Boolean columns found:", bool_cols)

BOOLEAN_COLUMN = "gsc_data_available"
avail = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE {BOOLEAN_COLUMN} IS TRUE) AS available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
avail["pct_available"] = (avail["available_rows"] / avail["total_rows"] * 100).round(1)
avail

Boolean columns found: ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,pct_available
0,9841378,3611061,36.7


In [29]:
feature_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_h1,
           SUM(gsc_clicks) AS clicks_h1,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr_h1,
           AVG(gsc_avg_position) AS avg_position_h1,
           COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS active_days_h1
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()
print(f"{len(feature_frame):,} content items with a usable feature row")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 content items with a usable feature row


,client_hash_id,content_hash_id,impressions_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,0.000000,3.659683,15
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,199.0,2.0,0.010050,4.086084,13
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,467.0,1.0,0.002141,4.449176,13
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,56.0,0.0,0.000000,6.600595,14
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,771.0,1.0,0.001297,1.883472,14


1. `impressions_h1` — summed daily impressions, days 1–15. Knowable same-day.
2. `clicks_h1` — same reasoning, summed daily clicks.
3. `ctr_h1` — clicks_h1 / impressions_h1. A ratio of two already-knowable sums.
4. `avg_position_h1` — mean daily rank, days 1–15. Logged per report day.
5. `active_days_h1` — count of days with impressions > 0 in the window. A count of days already past.

In [30]:
outcome = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_h2
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY 1, 2
""").df()
data = feature_frame.merge(outcome, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["impressions_h2"] < 0.8 * data["impressions_h1"]).astype(int)
print(f"{len(data):,} rows | declining rate: {data['is_declining'].mean():.1%}")

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
md = data.dropna(subset=honest_features + ["is_declining"])
X_tr, X_te, y_tr, y_te = train_test_split(md[honest_features], md["is_declining"],
                                            test_size=0.25, random_state=42, stratify=md["is_declining"])
honest_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"HONEST score — AUC: {honest_auc:.3f}")

# THE TRAP: add the label-derived column on purpose
leaky_features = honest_features + ["impressions_h2"]
ld = data.dropna(subset=leaky_features + ["is_declining"])
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(ld[leaky_features], ld["is_declining"],
                                                test_size=0.25, random_state=42, stratify=ld["is_declining"])
leaky_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_tr2, y_tr2)
leaky_auc = roc_auc_score(y_te2, leaky_model.predict_proba(X_te2)[:, 1])
print(f"LEAKY score — AUC: {leaky_auc:.3f}  <- jumps toward 1.0  (+{leaky_auc - honest_auc:.3f})")

# REMOVE THE LEAK, KEEP THE HONEST NUMBER
del leaky_features
print(f"Final reported score (honest features only): AUC {honest_auc:.3f}")
print("impressions_h2 excluded — it is the quantity the label threshold is built from.")

import json, os
os.makedirs("../outputs", exist_ok=True)
json.dump({"honest_auc": float(honest_auc), "leaky_auc": float(leaky_auc),
           "n_rows": int(len(data)), "declining_rate": float(data["is_declining"].mean())},
          open("../outputs/w03_contract_metrics.json", "w"), indent=2, sort_keys=True)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,513 rows | declining rate: 29.6%
HONEST score — AUC: 0.590
LEAKY score — AUC: 1.000  <- jumps toward 1.0  (+0.410)
Final reported score (honest features only): AUC 0.590
impressions_h2 excluded — it is the quantity the label threshold is built from.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
history_spread = con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_gsc, MAX(gsc_data_start) AS latest_gsc,
           MIN(ga4_data_start) AS earliest_ga4, MAX(ga4_data_start) AS latest_ga4,
           COUNT(*) AS n_clients
    FROM {TABLES['dim_clients']}
""").df()
history_spread
# cross-reference: Query 3 above already showed what share of March rows are gsc_data_available —
# that percentage is the quantitative face of this same limitation.


,earliest_gsc,latest_gsc,earliest_ga4,latest_ga4,n_clients
0,2025-01-27,2026-06-02,2025-10-29,2026-06-01,104


**Unbalanced history, GSC-only early rows.** `dim_clients.gsc_data_start` / `ga4_data_start`
vary per client, so March 2026 isn't equally "mid-panel" for everyone — and rows before a
client's `ga4_data_start` carry zero-filled GA4 columns rather than true zero engagement
(Haris's point: `0` is a verified result, a missing/flagged row is a collection gap, not the
same thing). This notebook doesn't correct for that difference yet.

In [32]:
history_spread = con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_gsc, MAX(gsc_data_start) AS latest_gsc,
           MIN(ga4_data_start) AS earliest_ga4, MAX(ga4_data_start) AS latest_ga4,
           COUNT(*) AS n_clients
    FROM {TABLES['dim_clients']}
""").df()
history_spread
# cross-reference: Query 3 above already showed what share of March rows are gsc_data_available —
# that percentage is the quantitative face of this same limitation.

,earliest_gsc,latest_gsc,earliest_ga4,latest_ga4,n_clients
0,2025-01-27,2026-06-02,2025-10-29,2026-06-01,104


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.